In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder
from RMSELoss import RMSELoss
import plotly.graph_objects as go
from datetime import datetime
import itertools
import kaleido 


# Getting Dataframe

In [2]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)




# Encode Categorical_Features

In [3]:
# Define the categorical features
categorical_features = ["Material"]

le = LabelEncoder()

for feature in categorical_features:
    x_train[feature] = le.fit_transform(x_train[feature])
    x_dev[feature] = le.transform(x_dev[feature])  

# Split Categorical_Features

In [4]:
# Drop categorical features to get the continuous features
x_train_numerical_features = x_train.drop(categorical_features, axis=1)
x_dev_numerical_features = x_dev.drop(categorical_features, axis=1)

# Seperate the categorical features
x_train_categorical_features = x_train[categorical_features]
x_dev_categorical_features = x_dev[categorical_features]

# Change df into Tensors

In [6]:
train_tensor = torch.tensor(x_train.to_numpy(), dtype=torch.float)
x_train_numer_tensor = torch.tensor(x_train_numerical_features.to_numpy(),dtype=torch.float)
x_dev_numer_tensor = torch.tensor(x_dev_numerical_features.to_numpy(),dtype=torch.float)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float)

dev_tensor = torch.tensor(x_dev.to_numpy(), dtype=torch.float)
x_train_categorical_features_tensor = torch.tensor(x_train_categorical_features.to_numpy(),dtype=torch.long)
x_dev_categorical_features_tensor = torch.tensor(x_dev_categorical_features.to_numpy(),dtype=torch.long)
y_dev_tensor = torch.tensor(y_dev.to_numpy(), dtype=torch.float)

# Create TensorDatasets for training and validation
train_ds = TensorDataset(
    x_train_categorical_features_tensor,
    x_train_numer_tensor,
    y_train_tensor
)
val_ds = TensorDataset(
    x_dev_categorical_features_tensor,
    x_dev_numer_tensor,
    y_dev_tensor
)
g = torch.Generator()
g.manual_seed(42)

# Create DataLoaders for training and validation
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator= g)
val_loader   = DataLoader(val_ds,   batch_size=32)



# Check Tensors


In [6]:
#torch.set_printoptions(sci_mode=False, precision=3)
#print(x_train_numer_tensor)

# Define Model

In [7]:
# categories is defined as a tuple, we only have one categorical feature "Material"
# the second parameter has to be empty for the model to work correctly
# in hard numbers this is displaying (1,)
model = FTTransformer(
    categories=(x_train_categorical_features.shape[1],),
    num_continuous=x_train_numerical_features.shape[1],
    dim=9,
    dim_out=1,
    depth=4,
    heads=4,
    attn_dropout=0.1,
    ff_dropout=0.1
)

# (Alternative) Initiate a saved Model


In [ ]:
#to be able to match the model, define the same parameters above in Define Model
#for example if the saved model has a dim of 9 the dim in define model has to be 9 as well

#saved models are in the folder "trained models"

#name of model
model_name = "model_FTTransformer_lr0.00025_9_1_4_4_.1_0.1__epoch15600"

#path to the saved model
path = f'trained_models/hyperparametertuning/{model_name}.pt'
model.load_state_dict(torch.load(path))

<All keys matched successfully>

# Define Training Epoch

In [8]:
def train_one_epoch(train_loader):
    total_loss = 0.0

    for x_cat, x_cont, y in train_loader:
        optimizer.zero_grad()

        # bring y to shape [B,1]
        y = y.unsqueeze(-1)

        # forward + backward + step
        pred = model(x_cat, x_cont)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        

    # return the average loss over ALL batches
    return total_loss / len(train_loader)

In [9]:
params = {
    "dim":          [2, 4, 8],
    "depth":        [2, 4],
    "heads":        [2, 4],
    "attn_dropout": [0.1, 0.3],
    "ff_dropout":   [0.1, 0.3],
    "lr":           [0.00025, 0.0003],
    "weight_decay": [0, 1e-5],
}

# Training

In [ ]:


# Get all combinations of parameter values
keys = list(params.keys())
combinations = itertools.product(*(params[k] for k in keys))

# Loop through each combination
for combo in combinations:
    config = dict(zip(keys, combo))

    model = FTTransformer(
        categories=(1,),
        num_continuous=8,
        dim=config["dim"],
        dim_out=1,
        depth=config["depth"],
        heads=config["heads"],
        attn_dropout=config["attn_dropout"],
        ff_dropout=config["ff_dropout"],
    )

    weight_decay = config["weight_decay"]
    criterion = RMSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=weight_decay)

 

    criterion = RMSELoss()

    #Changeable Parameters
    # ----------------------------------------------------#
    #Model description for saving
    model_description    = f'FTTransformer_{config["lr"]}_{config["dim"]}_1_{config["depth"]}_{config["heads"]}_{config["attn_dropout"]}_{config["ff_dropout"]}'

    # Number of epochs to train
    EPOCHS = 1000
    #-----------------------------------------------------#

    # Lists to store per‐epoch losses
    train_losses = []
    val_losses   = []

    best_vloss   = float('inf')
    last_ckpt  = None
    #timestamp for manual checkpoint selection
    timestamp    = datetime.now().strftime('%Y%m%d_%H%M%S')


    for epoch in range(EPOCHS):
        print(f'\nEPOCH {epoch+1}/{EPOCHS}')

        # --------------------
        # 1) TRAINING PHASE
        # --------------------
        model.train()
        avg_loss = train_one_epoch(train_loader)
        train_losses.append(avg_loss)
        print(f'train loss: {avg_loss:.4f}')

        # --------------------
        # 2) VALIDATION PHASE
        # --------------------
        model.eval()
    
        val_loss = 0.0
        with torch.no_grad():
            for x_cat, x_cont, y in val_loader:
                y = y.unsqueeze(-1)
                pred = model(x_cat, x_cont)
                val_loss += criterion(pred, y).item()

    

        avg_vloss = val_loss / len(val_loader)
        val_losses.append(avg_vloss)
    
        print(f'valid loss: {avg_vloss:.4f}')

        # --------------------
        # 3) CHECKPOINTING
        # --------------------
        # to sort through the checkpoints manually, add timestamp to the filename
        # remove avg_total < best_vloss constraint and os.remove(last_ckpt) if condition

        #_date_{timestamp}
        if (epoch + 1) % 100 == 0 and avg_vloss < best_vloss:
            if last_ckpt is not None:
                os.remove(last_ckpt)

            best_vloss = avg_vloss

            ckpt_path = f'trained_models/hyperparametertuning/model_{model_description}__epoch{epoch+1}.pt'
            torch.save(model.state_dict(), ckpt_path)
            last_ckpt = ckpt_path

        epochs = list(range(1, len(train_losses) + 1))

        fig = go.Figure()

        # Training loss trace
        fig.add_trace(go.Scatter(
        x=epochs,
        y=train_losses,
        mode="lines",
        name="Training Loss",
        line=dict(color="royalblue", width=2)
    ))

    # Validation loss trace
    fig.add_trace(go.Scatter(
        x=epochs,
        y=val_losses,
        mode="lines",
        name="Validation Loss",
        line=dict(color="firebrick", width=2, dash="dash")
    ))

    # Layout enhancements
    fig.update_layout(
        title=f"Training & Validation Loss over {EPOCHS} Epochs - {model_description}",
        xaxis_title="Epoch",
        yaxis_title="Loss",
        xaxis=dict(
            tickmode='array',
            tickvals=list(range(0, 16500, 500)),  # Show ticks every 100 epochs
            tickfont=dict(size=10)
        ),
        yaxis=dict(
            tickformat=".2e" if max(train_losses + val_losses) > 1e4 else ".4f",  # Dynamic formatting
            gridcolor="lightgray"
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        template="plotly_white",
        margin=dict(t=60, b=40)
    )

    fig.write_image(f"Graphs_Hyperparametertuning/{model_description}_loss_plot.png", width=1800, height=600)






EPOCH 1/100
train loss: 2963.8589
valid loss: 3021.7906

EPOCH 2/100
train loss: 2965.9057
valid loss: 3021.7838

EPOCH 3/100
train loss: 2963.9599
valid loss: 3021.7771

EPOCH 4/100
train loss: 2961.7617
valid loss: 3021.7702

EPOCH 5/100
train loss: 2976.3949
valid loss: 3021.7634

EPOCH 6/100
train loss: 2968.7692
valid loss: 3021.7568

EPOCH 7/100
train loss: 2962.6946
valid loss: 3021.7500

EPOCH 8/100
train loss: 2974.3295
valid loss: 3021.7434

EPOCH 9/100
train loss: 2960.8216
valid loss: 3021.7369

EPOCH 10/100
train loss: 2969.8942
valid loss: 3021.7305

EPOCH 11/100
train loss: 2963.6877
valid loss: 3021.4441

EPOCH 12/100
train loss: 2965.4862
valid loss: 3021.4290

EPOCH 13/100
train loss: 2960.1311
valid loss: 3020.9423

EPOCH 14/100
train loss: 2965.5701
valid loss: 3020.9328

EPOCH 15/100
train loss: 2960.4273
valid loss: 3020.9237

EPOCH 16/100
train loss: 2955.3829
valid loss: 3020.9144

EPOCH 17/100
train loss: 2980.2533
valid loss: 3020.9057

EPOCH 18/100
train los

KeyboardInterrupt: 